# 02 · FDIC Acquisition & Storage — Google Colab + Google Drive
**BFT65B · Final Individual Report · Part 1 (15 pts)**

Mục tiêu: tải **Financials 1984Q1–2026Q2 (170 quý)**, **Failures đầy đủ**, **Institutions đầy đủ** từ FDIC BankFind API; lưu dữ liệu trên Google Drive, kiểm tra theo số dòng API và schema, tải tiếp sau khi Colab ngắt, xây layout Parquet, benchmark CSV/Parquet, codec và phân vùng bằng các phép đo thực tế. Phiên bản này thêm schema integer chuẩn, kiểm tra API ID xuyên trang, hậu kiểm API total và benchmark tự giới hạn RAM. **History** là nguồn bổ trợ tùy chọn cho Part 2.

> **Hai loại lưu trữ:** `MyDrive/FDIC_Bank_Failure_Project/` để lưu dữ liệu + manifest **bền vững**. `/content/fdic_part1_bench/` để chạy benchmark trên local disk **tạm thời** của máy Colab (không đo thời gian mạng Google Drive). Kết quả đo CSV/JSON được xuất về Drive.

> **Cách chạy:** (1) đặt `RUN_MODE="PILOT"`, `Runtime → Run all`, kiểm tra `outputs/part1/acceptance_gates.csv`; (2) đổi thành `RUN_MODE="FULL"`, chạy lại. Không xóa `data_raw/` và manifest khi chuyển FULL. Nếu bị ngắt, chạy lại để resume theo từng quý và từng trang nguồn phụ.

**Nguồn & phạm vi:** [FDIC BankFind API](https://api.fdic.gov/banks/docs/). Ngày bắt đầu nghiên cứu: 1984Q1; ngày kết thúc: 2026Q2. Đề yêu cầu 3 bảng; History là tùy chọn. `schema/field gaps` ở giai đoạn lịch sử phải được ghi nhận, không tô thành đầy đủ. Các biến/công thức Derived từ Notebook 01 không được gửi lên API.

## 0. Install packages + mount Drive

Giữ thư mục Google Drive **Restricted**. Việc `drive.mount()` cho Colab quyền truy cập với tài khoản bạn cấp phép **không đòi hỏi công khai thư mục**. Colab có thể yêu cầu cấp quyền lại sau khi runtime ngắt.

In [ ]:
%pip -q install requests pandas pyarrow openpyxl pyyaml psutil

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FDIC_Bank_Failure_Project')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive project:', PROJECT_ROOT)

Mounted at /content/drive
Drive project: /content/drive/MyDrive/FDIC_Bank_Failure_Project


In [ ]:
import json
from pathlib import Path

root = Path('/content/drive/MyDrive/FDIC_Bank_Failure_Project')
manifest_path = root / 'data_raw/fdic/manifest_colab_v1.json'

assert manifest_path.exists(), "Không tìm thấy manifest!"

manifest_check = json.loads(
    manifest_path.read_text(encoding='utf-8')
)

financial_entries = {
    k: v for k, v in manifest_check['entries'].items()
    if k.startswith('financials:')
}

verified = sum(
    v.get('status') in ('VERIFIED', 'VERIFIED_SCHEMA_GAP')
    for v in financial_entries.values()
)

files_present = sum(
    (root / 'data_raw/fdic/financials' /
     f"REPDTE={k.split(':')[1]}.parquet").is_file()
    for k in financial_entries
)

print('Financials entries:', len(financial_entries))
print('Verified in manifest:', verified)
print('Parquet files present:', files_present)

Financials entries: 170
Verified in manifest: 170
Parquet files present: 170


## 1. Configuration — chỉ chỉnh ở cell này

- `PILOT`: 2023Q1–Q4, cộng với Failures + Institutions full endpoint (để thử end-to-end).
- `FULL`: 170 quý Financials + Failures + Institutions; có thể bật thêm History.
- `DOWNLOAD_HISTORY=False`: theo đúng 3 bảng bắt buộc; bật True nếu cần hỗ trợ kiểm tra merger/other exit trong Part 2.
- `VERIFIED_SCHEMA_GAP`: file đủ dòng và khóa, nhưng API không xuất hiện một hay nhiều trường đã yêu cầu; giữ dữ liệu và báo cáo riêng.
- `RUN_POST_DOWNLOAD_API_AUDIT=True`: sau acquisition, dùng request `limit=1` để kiểm tra lại `meta.total` cho các quý và bảng phụ. Đây là **audit riêng**, không tự tải lại/ghi đè file.
- `BENCHMARK_MEMORY_BUDGET_GIB`: kiểm soát RAM, tự lấy mẫu có thể tái lập trên nhiều quý khi FULL quá lớn. Benchmark sample được ghi rõ, không gắn nhãn FULL.
- Dữ liệu API có `id` ở wrapper thì bảo lưu trong `API_RECORD_ID` ở các bảng phụ để phát hiện trang trùng; nếu endpoint không cung cấp ID, báo `CHECK_MISSING_API_IDS` thay vì PASS.

In [ ]:
import csv, gc, hashlib, json, math, os, platform, random, shutil, statistics, sys, time
from collections import Counter
from datetime import date, datetime, timezone
from pathlib import Path

import requests
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import openpyxl
import yaml

# ---------- USER SETTINGS ----------
RUN_MODE = 'FULL'                     # 'PILOT' | 'FULL'
PILOT_YEAR = 2023
START_QUARTER = '19840331'
END_QUARTER   = '20260630'
DOWNLOAD_HISTORY = False               # optional source for Part 2
REFRESH_AUX = False                    # True to re-download existing aux snapshot pages
VERIFY_API_TOTAL_ON_RESUME = False     # True adds live API request per cached quarter
PAGE_SIZE = 5000
TIMEOUT = (15, 90)
MAX_RETRIES = 5
RETRY_PAUSE = 0.2
BENCHMARK_REPEATS = 5
RUN_BENCHMARK = True
BENCHMARK_MAX_ROWS = None              # Optional hard cap; None = auto from RAM budget
BENCHMARK_MEMORY_BUDGET_GIB = 1.5      # Arrow estimate × safety factor must fit this budget
BENCHMARK_RAM_SAFETY_FACTOR = 3.0      # Conservative allowance for copies/CSV conversion
BENCHMARK_RANDOM_SEED = 42            # Deterministic across-quarter sampling
RUN_POST_DOWNLOAD_API_AUDIT = (RUN_MODE == 'FULL')  # PILOT: optional; FULL: default enabled
STORAGE_CODEC = 'snappy'
# -----------------------------------

assert RUN_MODE in {'PILOT','FULL'}
assert BENCHMARK_REPEATS >= 1
assert BENCHMARK_MEMORY_BUDGET_GIB > 0
assert BENCHMARK_RAM_SAFETY_FACTOR >= 1
assert BENCHMARK_MAX_ROWS is None or BENCHMARK_MAX_ROWS >= 1
RAW = PROJECT_ROOT / 'data_raw' / 'fdic'
FIN_RAW = RAW / 'financials'
AUX_RAW = RAW / 'aux_pages'
CURATED = PROJECT_ROOT / 'data_storage' / 'financials' / f'year_partitioned_{RUN_MODE.lower()}'
REPORTS = PROJECT_ROOT / 'outputs' / 'part1'
MANIFEST_FILE = RAW / 'manifest_colab_v1.json'
LOCAL_BENCH = Path('/content/fdic_part1_bench')
for d in [FIN_RAW, AUX_RAW, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)
print('Mode:', RUN_MODE, '| Runtime:', sys.version.split()[0],
      '| Arrow:', pa.__version__, '| Drive:', PROJECT_ROOT)

Mode: FULL | Runtime: 3.13.18 | Arrow: 23.0.1 | Drive: /content/drive/MyDrive/FDIC_Bank_Failure_Project


## 2. Read the Excel Data Dictionary generated by Notebook 01

Đọc trực tiếp `FDIC_Data_Dictionary_Updated.xlsx` trên Drive. Các mã biến là cột đầu tiên của sheets `Financials`, `Failures`, `Institutions`, `History`; type được lấy từ 4 file YAML trong `metadata/` (tên có `(1)` cũng được chấp nhận). Không thay danh sách trường của Data Dictionary bằng một list nhập cứng ở Notebook 02.

In [ ]:
DICT_PATH = PROJECT_ROOT / 'FDIC_Data_Dictionary_Updated.xlsx'
if not DICT_PATH.is_file():
    raise FileNotFoundError(
        f'Chưa có {DICT_PATH}. Hãy chạy Notebook 01 trên Colab trước.')
WORKBOOK = openpyxl.load_workbook(DICT_PATH, read_only=True, data_only=True)
DATASETS = ['financials', 'failures', 'institutions'] + (['history'] if DOWNLOAD_HISTORY else [])

def extract_fields(sheet):
    if sheet not in WORKBOOK.sheetnames:
        raise ValueError(f'Missing Excel sheet: {sheet}')
    fields = []
    for row in WORKBOOK[sheet].iter_rows(min_row=2, values_only=True):
        if row and row[0] is not None:
            name = str(row[0]).strip().upper()
            if name and name not in fields:
                fields.append(name)
    if not fields:
        raise ValueError(f'No field codes found in {sheet}')
    return fields

FIELDS = {s: extract_fields(s.title()) for s in DATASETS}
if not {'CERT','REPDTE'} <= set(FIELDS['financials']):
    raise ValueError('Financials requires CERT and REPDTE')
for source in ('failures', 'institutions'):
    if 'CERT' not in FIELDS[source]:
        raise ValueError(f'{source} must contain CERT for Part 2 joins')

YAML_BASENAME = {
    'financials': 'risview_properties.yaml',
    'failures': 'failure_properties.yaml',
    'institutions': 'institution_properties.yaml',
    'history': 'history_properties.yaml',
}
def load_yaml_types(source):
    p = PROJECT_ROOT / 'metadata' / YAML_BASENAME[source]
    if not p.is_file():
        alias = p.with_name(p.stem + '(1).yaml')
        if alias.is_file():
            p = alias
    if not p.is_file():
        raise FileNotFoundError(f'Missing metadata YAML: {p}. Upload YAML to Drive/metadata/')
    tree = yaml.safe_load(p.read_text(encoding='utf-8'))
    props = tree.get('properties', {}).get('data', {}).get('properties', {})
    return {str(k).upper(): v.get('type', 'string') for k,v in props.items() if isinstance(v,dict)}

TYPE_MAP = {s: load_yaml_types(s) for s in DATASETS}

def dtype_for(source, code):
    if code == 'CERT':
        return pa.int64()
    if code == 'REPDTE':
        return pa.string()
    typ = TYPE_MAP[source].get(code)
    if typ == 'integer':
        return pa.int64()
    if typ == 'number':
        return pa.float64()
    if typ == 'boolean':
        return pa.bool_()
    return pa.string()

# Keep API wrapper `id` as an audit-only technical column, not a model feature.
# Failures / Institutions / optional History all receive a persistent ID when API supplies one.
SCHEMAS = {
    source: pa.schema(
        [(code, dtype_for(source, code)) for code in FIELDS[source]]
        + ([('API_RECORD_ID', pa.string())] if source != 'financials' else [])
    )
    for source in DATASETS
}
SIGNATURES = {s: hashlib.sha256(
    json.dumps([(x.name,str(x.type)) for x in SCHEMAS[s]], sort_keys=True).encode()
).hexdigest() for s in DATASETS}
DICTIONARY_HASH = hashlib.sha256(DICT_PATH.read_bytes()).hexdigest()
for s in DATASETS:
    missing_meta = sorted(set(FIELDS[s]) - set(TYPE_MAP[s]))
    if missing_meta:
        raise ValueError(f'{s} has field codes absent from YAML: {missing_meta}')
    print(s, len(FIELDS[s]), 'fields:', FIELDS[s])
WORKBOOK.close()

financials 47 fields: ['CERT', 'REPDTE', 'STNAME', 'ASSET', 'DEP', 'DEPDOM', 'COREDEP', 'DEPINS', 'DEPUNINS', 'BRO', 'FREPP', 'OTHBRF', 'CHBAL', 'SC', 'SCMV', 'EQ', 'INTAN', 'INTANGW', 'LNLSGR', 'LNLSNET', 'LNRE', 'LNRECONS', 'LNCOMRE', 'LNREMULT', 'LNRERES', 'LNCI', 'LNCON', 'LNATRES', 'NCLNLS', 'P3ASSET', 'P9ASSET', 'NAASSET', 'NTLNLS', 'ORE', 'NETINC', 'INTINC', 'EINTEXP', 'NONII', 'NONIX', 'ELNATR', 'ROA', 'ROE', 'NIMY', 'RBC1AAJ', 'RBC1RWAJ', 'RBCRWAJ', 'CBLRIND']
failures 10 fields: ['CERT', 'NAME', 'CITYST', 'FAILDATE', 'FAILYR', 'COST', 'QBFASSET', 'QBFDEP', 'RESTYPE', 'RESTYPE1']
institutions 9 fields: ['CERT', 'NAME', 'STNAME', 'BKCLASS', 'ACTIVE', 'ESTYMD', 'ENDEFYMD', 'CHANGEC1', 'INSDATE']


## 3. Research quarter scope & API client

`meta.total` lấy từ **response thực**, không nhập tay. API requests retry với 429/5xx, dừng nhanh khi sai tên field (400). Chia dữ liệu theo từng quý, không tải 1.68 triệu dòng vào RAM ngay khi acquisition. `sort_by=CERT` phù hợp cho Financials trong một quý (CERT phải duy nhất). Ở các bảng phụ, `CERT` hoặc `TRANSNUM` có thể không duy nhất: notebook kiểm tra API record ID xuyên trang và ghi cảnh báo nếu khóa sắp xếp trùng ngay ranh giới trang; việc này giảm rủi ro nhưng không biến offset pagination thành snapshot cố định.

In [ ]:

def quarter_ends(first='19840331', last='20260630'):
    first_dt = datetime.strptime(first, '%Y%m%d').date()
    last_dt = datetime.strptime(last, '%Y%m%d').date()
    result = []
    for year in range(first_dt.year, last_dt.year + 1):
        for month, day in [(3, 31), (6, 30), (9, 30), (12, 31)]:
            cur = date(year, month, day)
            if first_dt <= cur <= last_dt:
                result.append(cur.strftime('%Y%m%d'))
    return result


ALL_QUARTERS = quarter_ends(START_QUARTER, END_QUARTER)
assert len(ALL_QUARTERS) == 170

PILOT_QUARTERS = [
    q for q in ALL_QUARTERS
    if q[:4] == str(PILOT_YEAR)
]
assert len(PILOT_QUARTERS) == 4

TARGET_QUARTERS = (
    PILOT_QUARTERS if RUN_MODE == 'PILOT'
    else ALL_QUARTERS
)

print(
    'Research:',
    ALL_QUARTERS[0], '→', ALL_QUARTERS[-1],
    '=', len(ALL_QUARTERS), 'quarters'
)
print('Target:', len(TARGET_QUARTERS), 'quarters')


class NonRetryableAPIError(RuntimeError):
    pass


class FDICClient:
    base = 'https://api.fdic.gov/banks'
    retry_codes = {408, 429, 500, 502, 503, 504}

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'FDIC-Finance-Academic-Research/1.0'
        })

    def page(
        self,
        source,
        fields,
        *,
        offset=0,
        filters=None,
        sort_by='CERT',
        limit=None
    ):
        effective_limit = PAGE_SIZE if limit is None else int(limit)

        if not 1 <= effective_limit <= 10_000:
            raise ValueError(
                'API page limit must be between 1 and 10000'
            )

        params = {
            'fields': ','.join(fields),
            'sort_by': sort_by,
            'sort_order': 'ASC',
            'limit': effective_limit,
            'offset': offset,
            'format': 'json'
        }

        if filters is not None:
            params['filters'] = filters

        url = f'{self.base}/{source}'

        for attempt in range(MAX_RETRIES):
            try:
                r = self.session.get(
                    url,
                    params=params,
                    timeout=TIMEOUT
                )

                if r.status_code == 200:
                    payload = r.json()

                    meta = payload.get('meta')
                    raw = payload.get('data')

                    if (
                        not isinstance(meta, dict)
                        or not isinstance(raw, list)
                        or 'total' not in meta
                    ):
                        raise RuntimeError(
                            f'Unexpected response structure from {source}'
                        )

                    total = int(meta['total'])
                    rows = []

                    for item in raw:
                        row = item.get('data', item)

                        if not isinstance(row, dict):
                            raise RuntimeError(
                                'Unexpected API row'
                            )

                        row = dict(row)

                        # FIX: Extract API record ID correctly.
                        # FDIC API stores ID inside item["data"]["ID"].
                        if source != 'financials':

                            rid = row.get('ID')

                            # Fallback for alternative API structures
                            if rid is None and isinstance(item, dict):
                                rid = item.get('id')

                            row['__api_id'] = (
                                None if rid is None
                                else str(rid)
                            )

                        rows.append(row)

                    if (
                        len(rows) > effective_limit
                        or (offset < total and not rows)
                    ):
                        raise RuntimeError(
                            f'Invalid page size: '
                            f'offset={offset}, total={total}'
                        )

                    time.sleep(RETRY_PAUSE)

                    return rows, total

                if r.status_code not in self.retry_codes:
                    raise NonRetryableAPIError(
                        f'HTTP {r.status_code} {source}: '
                        f'{r.text[:500]} | requested={params}'
                    )

                retry_after = r.headers.get('Retry-After', '')

                delay = (
                    float(retry_after)
                    if retry_after.isdigit()
                    else min(30, 2**attempt)
                )

            except (
                requests.exceptions.Timeout,
                requests.exceptions.ConnectionError
            ):
                delay = min(30, 2**attempt)

            if attempt == MAX_RETRIES - 1:
                break

            print(
                f'  Retry {source}: '
                f'{attempt + 1}/{MAX_RETRIES}'
            )

            time.sleep(delay)

        raise RuntimeError(
            f'API retry exhausted: '
            f'{source} offset={offset}, filters={filters}'
        )


api = FDICClient()


Research: 19840331 → 20260630 = 170 quarters
Target: 170 quarters


In [ ]:
for source in ['failures', 'institutions']:
    rows, total = api.page(
        source,
        fields=['ID', 'CERT', 'NAME'],
        limit=5,
        sort_by='CERT'
    )

    ids = [row.get('__api_id') for row in rows]

    print(f'\n{source.upper()}')
    print('API total:', total)
    print('API IDs:', ids)
    print('Missing IDs:', sum(x is None for x in ids))


FAILURES
API total: 4117
API IDs: ['584', '1052', '1419', '4088', '1998']
Missing IDs: 0

INSTITUTIONS
API total: 27835
API IDs: ['4', '6', '8', '9', '10']
Missing IDs: 0


## 4. Resumable integrity layer

**Điều kiện để skip một quý:** manifest v1 ghi nhận `VERIFIED`/`VERIFIED_SCHEMA_GAP`, SHA-256 file trùng, Arrow schema khớp, record count khớp API total lúc tải, mọi `REPDTE` đúng quý, `CERT` unique và không null. Bảng phụ còn lưu `API_RECORD_ID`; cache cũ thiếu ID sẽ được tải lại vì schema signature đổi. Chỉ khi đạt toàn bộ mới giữ file cũ. Mất kết nối giữa quý → tải lại **quý đó**, không tải lại các quý đã xác minh.

Ghi chú kỹ thuật: dữ liệu API có thể thay đổi sau ngày tải. Bật `VERIFY_API_TOTAL_ON_RESUME=True` nếu muốn kiểm tra lại tổng số dòng qua API; kiểm tra total **không** phát hiện mọi chỉnh sửa nội dung từ phía FDIC.

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def save_json_atomic(obj,path):
    path = Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp = path.with_name(path.name + '.partial')
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    os.replace(tmp,path)

def read_manifest():
    if MANIFEST_FILE.is_file():
        obj = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
        if obj.get('version') != 1:
            raise ValueError('Manifest has incompatible version; do not auto-trust old cache.')
        return obj
    return {'version':1, 'entries':{}}

manifest = read_manifest()
def commit_manifest():
    manifest['last_updated_utc'] = utc_now()
    save_json_atomic(manifest, MANIFEST_FILE)

def normalize_date(value):
    if value is None:
        return None
    txt = str(value).strip()
    if txt.endswith('.0'):
        txt=txt[:-2]
    txt = txt.replace('-','').replace('/','')
    return txt[:8] if len(txt)>=8 else txt

from decimal import Decimal, InvalidOperation

def exact_integer(value, context):
    # Never silently truncate 12.75 → 12, or convert a large identifier via float.
    try:
        parsed = Decimal(str(value))
    except (InvalidOperation, ValueError) as exc:
        raise ValueError(f'Invalid integer at {context}: {value!r}') from exc
    if not parsed.is_finite() or parsed != parsed.to_integral_value():
        raise ValueError(f'Non-integral value at {context}: {value!r}')
    return int(parsed)

def normalize_table(source, rows):
    schema = SCHEMAS[source]
    record_rows = []
    for r in rows:
        converted = {}
        for field in schema:
            name = field.name
            x = r.get('__api_id') if name == 'API_RECORD_ID' else r.get(name)
            if name == 'REPDTE':
                x=normalize_date(x)
            if x is None or x == '' or x == 'null':
                x = None
            elif pa.types.is_integer(field.type):
                x = exact_integer(x, f'{source}.{name}')
            elif pa.types.is_floating(field.type):
                try:
                    x = float(x)
                    if not math.isfinite(x): x = None
                except (TypeError,ValueError):
                    raise ValueError(f'Unexpected numeric value {source}.{name}: {x!r}')
            elif pa.types.is_boolean(field.type):
                x = str(x).strip().lower() in {'1','true','yes'}
            else:
                x = str(x)
            converted[name] = x
        record_rows.append(converted)
    return pa.Table.from_pylist(record_rows,schema=schema)

def write_verified_parquet(table,path,compression=STORAGE_CODEC):
    path = Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp = path.with_name(path.name+'.partial')
    try:
        pq.write_table(table,tmp,compression=compression,version='2.6')
        check = pq.ParquetFile(tmp)
        if check.metadata.num_rows != table.num_rows or not check.schema_arrow.equals(table.schema,check_metadata=False):
            raise RuntimeError('Parquet readback metadata mismatch')
        os.replace(tmp,path)
    finally:
        if tmp.exists(): tmp.unlink()

def financial_path(q):
    return FIN_RAW / f'REPDTE={q}.parquet'

def verify_financial(table,quarter):
    if table.num_rows == 0:
        raise ValueError('Empty financial quarter')
    if not table.schema.equals(SCHEMAS['financials'], check_metadata=False):
        raise ValueError('Arrow schema mismatch')
    dates = table.column('REPDTE').to_pylist()
    certs = table.column('CERT').to_pylist()
    if any(x != quarter for x in dates):
        raise ValueError('REPDTE outside requested quarter')
    if any(x is None for x in certs) or len(set(certs)) != len(certs):
        raise ValueError('Null or duplicate CERT within quarter')

def verified_cache(entry,path,source,quarter=None):
    if not entry or entry.get('status') not in {'VERIFIED','VERIFIED_SCHEMA_GAP'}:
        return False,'manifest not verified'
    if entry.get('signature') != SIGNATURES[source]:
        return False,'schema or selected fields changed'
    if not path.exists() or sha256_file(path) != entry.get('sha256'):
        return False,'file absent or checksum mismatch'
    try:
        pf = pq.ParquetFile(path)
        if pf.metadata.num_rows != entry.get('rows') or pf.metadata.num_rows != entry.get('api_total'):
            return False,'record count mismatch'
        if not pf.schema_arrow.equals(SCHEMAS[source],check_metadata=False):
            return False,'schema mismatch'
        if quarter:
            verify_financial(pf.read(),quarter)
        return True,'pass'
    except Exception as exc:
        return False,str(exc)[:100]

print('Manifest entries loaded:',len(manifest['entries']))

Manifest entries loaded: 179


## 5. Financials downloader: PILOT / FULL cùng một pipeline

Mỗi quý gọi API theo pages (`offset`, `limit`); tổng số dòng phải khớp `meta.total` trên **tất cả** các trang. Mỗi quý ghi Parquet Snappy lên Drive và manifest checkpoint. Nếu API trả 0 record: `SOURCE_GAP`; nếu một hay nhiều biến không xuất hiện trong response: `VERIFIED_SCHEMA_GAP` (không gian lận bằng cách tự tuyên bố có đủ biến). `VERIFIED_SCHEMA_GAP` vẫn có thể dùng trong phân tích với các ô null, nhưng cần nêu trong report.

In [ ]:
def get_full_financial_quarter(q,first,total):
    rows = list(first)
    observed = set().union(*(set(r) for r in first)) if first else set()
    while len(rows) < total:
        chunk,new_total = api.page('financials',FIELDS['financials'],
                                   filters=f'REPDTE:{q}', offset=len(rows),sort_by='CERT')
        if new_total != total:
            raise RuntimeError(f'API total drift for {q}: {total} → {new_total}')
        rows.extend(chunk)
        observed.update(*(set(r) for r in chunk))
    if len(rows) != total:
        raise RuntimeError(f'{q}: expected {total}, received {len(rows)}')
    return rows,observed

def acquire_financial(q,index,count):
    key = 'financials:'+q
    dest = financial_path(q)
    old = manifest['entries'].get(key)
    okay,why = verified_cache(old,dest,'financials',quarter=q)
    if okay:
        if VERIFY_API_TOTAL_ON_RESUME:
            _,live_total = api.page('financials',FIELDS['financials'],filters=f'REPDTE:{q}',sort_by='CERT')
            okay = old.get('api_total') == live_total
        if okay:
            print(f'[{index}/{count}] CACHE {q}: {old["rows"]:,} rows')
            return old
    if old:
        print(f'[{index}/{count}] REDOWNLOAD {q}: {why}')
    entry = {'source':'financials','quarter':q,'signature':SIGNATURES['financials'],
             'dictionary_hash':DICTIONARY_HASH,'status':'ERROR', 'api_total':None,'rows':0,
             'started_at':utc_now()}
    t0=time.perf_counter()
    try:
        first,total = api.page('financials',FIELDS['financials'],filters=f'REPDTE:{q}',
                               offset=0,sort_by='CERT')
        entry['api_total']=total
        if total == 0:
            entry.update(status='SOURCE_GAP',error='meta.total=0')
        else:
            rows,obs = get_full_financial_quarter(q,first,total)
            missing = sorted(set(FIELDS['financials'])-obs)
            table = normalize_table('financials',rows)
            verify_financial(table,q)
            write_verified_parquet(table,dest)
            entry.update(status='VERIFIED_SCHEMA_GAP' if missing else 'VERIFIED',
                         rows=table.num_rows,sha256=sha256_file(dest),
                         missing_fields=missing, file=str(dest.relative_to(PROJECT_ROOT)))
            good,why = verified_cache(entry,dest,'financials',quarter=q)
            if not good: raise RuntimeError('post-write failed: '+why)
            print(f'[{index}/{count}] {entry["status"]} {q}: {len(rows):,}/{total:,} '
                  f'| missing selected fields: {len(missing)}')
    except NonRetryableAPIError:
        entry.update(status='ERROR',error='Non-retryable API response (see traceback)')
        raise
    except Exception as exc:
        entry.update(status='ERROR',error=f'{type(exc).__name__}: {str(exc)[:400]}')
        print(f'[{index}/{count}] ERROR {q}: {entry["error"]}')
    finally:
        entry['duration_s']=round(time.perf_counter()-t0,2)
        entry['checked_at']=utc_now()
        manifest['entries'][key]=entry
        commit_manifest()
    return entry

## 6. Complete Failures + Institutions; optional History

Các bảng này tải **toàn endpoint, không áp filter năm**, phân trang vào snapshot; tất cả pages phải khớp `meta.total`. Trường `data.ID` của từng bản ghi Failures/Institutions được giữ dưới tên `API_RECORD_ID`, kiểm tra trùng và thiếu qua các trang, kể cả trang được reuse từ cache. Snapshot name chứa query signature + total, nên nếu API total thay đổi không trộn với những pages cũ. History có thể lớn: chỉ tải khi `DOWNLOAD_HISTORY=True`.

In [ ]:
SORT_KEYS = {'failures': 'CERT', 'institutions': 'CERT', 'history': 'TRANSNUM'}

def aux_page_path(source, snapshot, idx):
    return AUX_RAW / source / f'snapshot={snapshot}' / f'page_{idx:05d}.parquet'

def acquire_aux(source):
    # Explicitly request the nested FDIC data.ID for auxiliary record audits.
    # Keep FIELDS and SCHEMAS unchanged; ID is stored as API_RECORD_ID.
    fields = list(FIELDS[source])
    if source in ('failures', 'institutions') and 'ID' not in fields:
        fields.append('ID')
    sort_by = SORT_KEYS[source]
    first, total = api.page(source, fields, sort_by=sort_by)
    query = json.dumps({'source': source, 'fields': fields, 'sort': sort_by,
                        'page_size': PAGE_SIZE, 'total': total,
                        'signature': SIGNATURES[source]}, sort_keys=True)
    snapshot = hashlib.sha256(query.encode()).hexdigest()[:18]
    num_pages = math.ceil(total / PAGE_SIZE)
    summary_key = f'aux:{source}:summary'

    observed = set()
    seen_api_ids = set()
    duplicate_id_examples = []
    duplicate_api_ids = 0
    missing_api_ids = 0
    identities = []
    total_rows = 0
    previous_sort_last = None
    page_boundary_sort_ties = 0

    print(f'{source}: API total={total:,}; pages={num_pages}; snapshot={snapshot}')
    if total == 0:
        manifest['entries'][summary_key] = {
            'source': source, 'status': 'SOURCE_GAP', 'api_total': 0,
            'rows': 0, 'snapshot': snapshot, 'checked_at': utc_now()
        }
        commit_manifest()
        return manifest['entries'][summary_key]

    for idx in range(num_pages):
        page_key = f'aux:{source}:{snapshot}:{idx:05d}'
        path = aux_page_path(source, snapshot, idx)
        expected = min(PAGE_SIZE, total - idx * PAGE_SIZE)
        cached = manifest['entries'].get(page_key)
        good, _ = verified_cache(cached, path, source)
        good = good and not REFRESH_AUX and cached['rows'] == expected

        # A hash-valid old Parquet page may still have 100% missing IDs.
        # Never trust such a cached page for Failures or Institutions.
        if good and source in ('failures', 'institutions'):
            try:
                id_col = pq.read_table(path, columns=['API_RECORD_ID']).column('API_RECORD_ID')
                cached_ids = id_col.to_pylist()
                if any(v is None or not str(v).strip() for v in cached_ids):
                    print(f'  {source}: page {idx+1} cached IDs missing; refreshing page')
                    good = False
            except Exception as exc:
                print(f'  {source}: cannot validate cached IDs ({exc}); refreshing page')
                good = False

        if good:
            tab = pq.ParquetFile(path).read()
            observed.update(cached.get('observed_fields', []))
        else:
            chunk, total_from_page = ((first, total) if idx == 0 else
                api.page(source, fields, offset=idx * PAGE_SIZE, sort_by=sort_by))
            if total_from_page != total or len(chunk) != expected:
                raise RuntimeError(f'{source}: incomplete or changing API page {idx+1}/{num_pages}')
            got = set().union(*(set(r) for r in chunk)) if chunk else set()
            observed.update(got)
            tab = normalize_table(source, chunk)
            write_verified_parquet(tab, path)
            item = {'source': source, 'status': 'VERIFIED',
                    'signature': SIGNATURES[source], 'dictionary_hash': DICTIONARY_HASH,
                    'rows': tab.num_rows, 'api_total': tab.num_rows,
                    'sha256': sha256_file(path), 'observed_fields': sorted(got),
                    'snapshot': snapshot, 'page': idx,
                    'file': str(path.relative_to(PROJECT_ROOT))}
            manifest['entries'][page_key] = item
            commit_manifest()

        # This audit runs on cached AND newly downloaded pages. A cached page
        # cannot bypass cross-page duplicate detection.
        page_ids = tab.column('API_RECORD_ID').to_pylist()
        for record_id in page_ids:
            if record_id is None or not str(record_id).strip():
                missing_api_ids += 1
            elif record_id in seen_api_ids:
                duplicate_api_ids += 1
                if len(duplicate_id_examples) < 5:
                    duplicate_id_examples.append(record_id)
            else:
                seen_api_ids.add(record_id)

        # FDIC API supports sort_by one field; tied values at page boundaries
        # mean offset ordering is not provably unique (even if IDs are unique).
        if sort_by in tab.column_names and tab.num_rows:
            sort_values = tab.column(sort_by)
            first_sort = sort_values[0].as_py()
            last_sort = sort_values[-1].as_py()
            if previous_sort_last is not None and first_sort == previous_sort_last:
                page_boundary_sort_ties += 1
            previous_sort_last = last_sort

        if source == 'institutions':
            identities.extend(tab.column('CERT').to_pylist())
        total_rows += tab.num_rows
        if (idx + 1) % 10 == 0 or idx == num_pages - 1:
            print(f'  {source}: {idx+1}/{num_pages} pages; {total_rows:,} rows')

    missing_fields = sorted(set(fields) - observed)
    cert_unique = (all(cert is not None for cert in identities)
                   and len(set(identities)) == len(identities)) if identities else True

    if duplicate_api_ids:
        api_id_check = 'FAIL_DUPLICATE_IDS'
    elif missing_api_ids:
        api_id_check = 'CHECK_MISSING_IDS'
    else:
        api_id_check = 'PASS'

    if total_rows != total:
        status = 'ERROR_RECORD_COUNT'
    elif duplicate_api_ids:
        status = 'ERROR_DUPLICATE_API_IDS'
    elif not cert_unique:
        status = 'ERROR_DUPLICATE_CERT'
    elif missing_api_ids:
        status = 'CHECK_MISSING_API_IDS'
    else:
        status = 'VERIFIED_SCHEMA_GAP' if missing_fields else 'VERIFIED'

    summary = {'source': source, 'status': status, 'api_total': total,
               'rows': total_rows, 'pages': num_pages, 'snapshot': snapshot,
               'sort_by': sort_by, 'missing_fields': missing_fields,
               'api_id_check': api_id_check, 'id_check': api_id_check,
               'unique_api_ids': len(seen_api_ids),
               'duplicate_api_ids': duplicate_api_ids,
               'missing_api_ids': missing_api_ids,
               'duplicate_id_examples': duplicate_id_examples,
               'page_boundary_sort_ties': page_boundary_sort_ties,
               'pagination_warning': ('Sort key repeats across page boundaries; '
                                      'snapshot consistency not guaranteed')
                   if page_boundary_sort_ties else '',
               'checked_at': utc_now(),
               'snapshot_dir': str((AUX_RAW / source / f'snapshot={snapshot}')
                                   .relative_to(PROJECT_ROOT))}
    manifest['entries'][summary_key] = summary
    commit_manifest()
    print(f'{source}: {status} {total_rows:,}/{total:,} rows; '
          f'IDs unique={len(seen_api_ids):,}, duplicates={duplicate_api_ids}, '
          f'missing={missing_api_ids}, sort-boundary ties={page_boundary_sort_ties}')
    return summary

## 7. Execute acquisition — real API download

Nếu đây là lần đầu, Colab có thể tải khá lâu. **Không cần chạy hết FULL trong một phiên duy nhất**: manifest và file đã xác minh lưu trên Drive. Nếu notebook bị ngắt, chạy lại từ đầu, downloader sẽ skip các quý/pages tốt. Nếu gặp HTTP 400 do tên trường, notebook dừng sớm để không lặp lỗi trên 170 quý.

In [ ]:
# Check selected fields on an existing quarter to fail fast on invalid API requests.
probe_q='20231231'
probe,probe_total=api.page('financials',FIELDS['financials'],filters=f'REPDTE:{probe_q}',sort_by='CERT')
assert probe_total > 0, 'Preflight query unexpectedly returned no rows'
print('FDIC API preflight passed, 2023Q4 meta.total =', probe_total)

run_started = time.perf_counter()
for index,q in enumerate(TARGET_QUARTERS,1):
    acquire_financial(q,index,len(TARGET_QUARTERS))
for src in (['failures','institutions'] + (['history'] if DOWNLOAD_HISTORY else [])):
    try:
        acquire_aux(src)
    except NonRetryableAPIError:
        raise
    except Exception as exc:
        manifest['entries'][f'aux:{src}:summary']={
            'source':src, 'status':'ERROR','error':f'{type(exc).__name__}: {exc}',
            'checked_at':utc_now()}
        commit_manifest()
        print(f'ERROR {src}: {exc}')
print('Download stage finished in',round((time.perf_counter()-run_started)/60,2),'minutes')

FDIC API preflight passed, 2023Q4 meta.total = 4658
[1/170] CACHE 19840331: 17,930 rows
[2/170] CACHE 19840630: 17,935 rows
[3/170] CACHE 19840930: 17,946 rows
[4/170] CACHE 19841231: 17,949 rows
[5/170] CACHE 19850331: 17,968 rows
[6/170] CACHE 19850630: 17,853 rows
[7/170] CACHE 19850930: 18,058 rows
[8/170] CACHE 19851231: 18,082 rows
[9/170] CACHE 19860331: 18,133 rows
[10/170] CACHE 19860630: 18,068 rows
[11/170] CACHE 19860930: 18,051 rows
[12/170] CACHE 19861231: 17,930 rows
[13/170] CACHE 19870331: 17,795 rows
[14/170] CACHE 19870630: 17,660 rows
[15/170] CACHE 19870930: 17,560 rows
[16/170] CACHE 19871231: 17,384 rows
[17/170] CACHE 19880331: 17,204 rows
[18/170] CACHE 19880630: 17,057 rows
[19/170] CACHE 19880930: 16,822 rows
[20/170] CACHE 19881231: 16,632 rows
[21/170] CACHE 19890331: 16,505 rows
[22/170] CACHE 19890630: 16,444 rows
[23/170] CACHE 19890930: 16,296 rows
[24/170] CACHE 19891231: 16,152 rows
[25/170] CACHE 19900331: 16,011 rows
[26/170] CACHE 19900630: 15,762 

## 8. Post-download API audit (FULL: tự động; PILOT: tùy chọn)

Sau acquisition, kiểm tra lại `meta.total` của **mỗi quý** và **Failures/Institutions** bằng API request `limit=1`. Báo cáo `post_download_api_audit.csv` ghi `MATCH`, `DRIFT`, `NO_LOCAL_VERIFIED_DATA`, `ERROR`, ... và **không** ghi đè hoặc tải lại Parquet. Nếu có `DRIFT`, cần quyết định tải lại quý/nguồn đó sau khi kiểm tra thay đổi của FDIC.

Đây là kiểm tra **record-count drift**, không chứng minh nội dung toàn bộ nguồn không đổi. Kết quả audit cũng không giải quyết vấn đề sort key không duy nhất của offset pagination. `API_RECORD_ID` trên các nguồn phụ cung cấp bằng chứng bổ sung để kiểm tra tính duy nhất.


In [ ]:
post_download_api_audit = []
if RUN_POST_DOWNLOAD_API_AUDIT:
    print('Post-download API audit: checking live totals with limit=1 ...')

for index, q in enumerate(TARGET_QUARTERS, 1):
    entry = manifest['entries'].get('financials:' + q, {})
    saved_status = entry.get('status', 'NOT_ATTEMPTED')
    saved_total = entry.get('api_total')
    row = {'source': 'financials', 'quarter': q, 'manifest_status': saved_status,
           'manifest_api_total': saved_total, 'local_rows': entry.get('rows'),
           'live_api_total': None, 'audit_result': 'NOT_RUN', 'error': '',
           'checked_at': None}
    if RUN_POST_DOWNLOAD_API_AUDIT:
        try:
            _, live_total = api.page('financials', ['CERT', 'REPDTE'],
                                     filters=f'REPDTE:{q}', sort_by='CERT', limit=1)
            row['live_api_total'] = live_total
            if saved_status not in {'VERIFIED', 'VERIFIED_SCHEMA_GAP'}:
                row['audit_result'] = 'NO_LOCAL_VERIFIED_DATA'
            elif saved_total != live_total:
                row['audit_result'] = 'DRIFT'
            else:
                row['audit_result'] = 'MATCH'
        except Exception as exc:
            row['audit_result'] = 'ERROR'
            row['error'] = f'{type(exc).__name__}: {str(exc)[:200]}'
        row['checked_at'] = utc_now()
        if index % 25 == 0 or index == len(TARGET_QUARTERS):
            print(f'  Quarter audit {index}/{len(TARGET_QUARTERS)}')
    post_download_api_audit.append(row)

for source in (['failures', 'institutions'] + (['history'] if DOWNLOAD_HISTORY else [])):
    entry = manifest['entries'].get(f'aux:{source}:summary', {})
    saved_status = entry.get('status', 'NOT_ATTEMPTED')
    saved_total = entry.get('api_total')
    row = {'source': source, 'quarter': '', 'manifest_status': saved_status,
           'manifest_api_total': saved_total, 'local_rows': entry.get('rows'),
           'live_api_total': None, 'audit_result': 'NOT_RUN', 'error': '',
           'checked_at': None}
    if RUN_POST_DOWNLOAD_API_AUDIT:
        try:
            _, live_total = api.page(source, [FIELDS[source][0]],
                                     sort_by=SORT_KEYS[source], limit=1)
            row['live_api_total'] = live_total
            if saved_status not in {'VERIFIED', 'VERIFIED_SCHEMA_GAP'}:
                row['audit_result'] = 'NO_LOCAL_VERIFIED_DATA'
            elif saved_total != live_total:
                row['audit_result'] = 'DRIFT'
            else:
                row['audit_result'] = 'MATCH'
        except Exception as exc:
            row['audit_result'] = 'ERROR'
            row['error'] = f'{type(exc).__name__}: {str(exc)[:200]}'
        row['checked_at'] = utc_now()
    post_download_api_audit.append(row)

audit_df = pd.DataFrame(post_download_api_audit)
post_audit_all_match = (RUN_POST_DOWNLOAD_API_AUDIT
                        and not audit_df.empty
                        and bool(audit_df['audit_result'].eq('MATCH').all()))
print('Post-download audit:', audit_df['audit_result'].value_counts().to_dict(),
      '| all match:', post_audit_all_match)


Post-download API audit: checking live totals with limit=1 ...
  Quarter audit 25/170
  Quarter audit 50/170
  Quarter audit 75/170
  Quarter audit 100/170
  Quarter audit 125/170
  Retry financials: 1/5
  Quarter audit 150/170
  Quarter audit 170/170
Post-download audit: {'MATCH': 172} | all match: True


## 9. Real completeness reports: 170-quarter coverage + record reconciliation + source audits

`quarter_coverage.csv` liệt kê đủ **170 quý**, nhưng `NOT_ATTEMPTED` không được tính là đã tải. `VERIFIED_SCHEMA_GAP` nghĩa là số dòng/khóa hoàn chỉnh nhưng thiếu trường có thể do lịch sử báo cáo hoặc API. Mọi kết luận ghi rõ phân biệt **temporal scope**, **observed API record completeness**, **field completeness**.

In [ ]:
coverage=[]
verified_quarters=[]
reconciliation=[]
for i, q in enumerate(ALL_QUARTERS, start=1):

    if i == 1 or i % 10 == 0:
        print(
            f"Validating cached Parquet: {i}/{len(ALL_QUARTERS)}"
            f" | Quarter: {q}",
            flush=True
        )
    entry=manifest['entries'].get('financials:'+q,{})
    status=entry.get('status','NOT_ATTEMPTED')
    why=entry.get('error','')
    if status in {'VERIFIED','VERIFIED_SCHEMA_GAP'}:
        good,why_check=verified_cache(entry,financial_path(q),'financials',quarter=q)
        if not good:
            status='INVALID_CACHE'
            why=why_check
        else:
            verified_quarters.append(q)
    coverage.append({'quarter':q,'year':int(q[:4]),'requested_in_this_run':q in TARGET_QUARTERS,
                     'status':status,'rows':entry.get('rows'),
                     'api_total_at_download':entry.get('api_total'),
                     'missing_fields':';'.join(entry.get('missing_fields',[])),
                     'checked_at':entry.get('checked_at'), 'notes':why})
    if q in TARGET_QUARTERS:
        reconciliation.append({'quarter':q,'status':status,'rows':entry.get('rows'),
                               'api_total_at_download':entry.get('api_total'),
                               'record_count_match':status in {'VERIFIED','VERIFIED_SCHEMA_GAP'} and entry.get('rows')==entry.get('api_total'),
                               'schema_fields_all_observed':status=='VERIFIED',
                               'cache_hash_validated':status in {'VERIFIED','VERIFIED_SCHEMA_GAP'},
                               'missing_fields':';'.join(entry.get('missing_fields',[])),
                               'error':why})
coverage_df=pd.DataFrame(coverage)
reconciliation_df=pd.DataFrame(reconciliation)
source_data=[{
    'source':'financials','scope':RUN_MODE,
    'api_total_at_download':int(pd.to_numeric(reconciliation_df.api_total_at_download,errors='coerce').fillna(0).sum()),
    'downloaded_rows':int(pd.to_numeric(reconciliation_df.rows,errors='coerce').fillna(0).sum()),
    'target_quarters':len(TARGET_QUARTERS),'verified_quarters':int(reconciliation_df.status.isin(['VERIFIED','VERIFIED_SCHEMA_GAP']).sum()),
    'status':'VERIFIED' if reconciliation_df.status.isin(['VERIFIED','VERIFIED_SCHEMA_GAP']).all() else 'INCOMPLETE'
}]
for src in (['failures','institutions']+(['history'] if DOWNLOAD_HISTORY else [])):
    info=manifest['entries'].get(f'aux:{src}:summary',{})
    source_data.append({'source':src,'scope':'entire endpoint',
        'api_total_at_download':info.get('api_total'),
        'downloaded_rows':info.get('rows'), 'target_quarters':None,'verified_quarters':None,
        'status':info.get('status','NOT_ATTEMPTED'),
        'api_id_check': info.get('api_id_check'),
        'unique_api_ids': info.get('unique_api_ids'),
        'duplicate_api_ids': info.get('duplicate_api_ids'),
        'missing_api_ids': info.get('missing_api_ids'),
        'page_boundary_sort_ties': info.get('page_boundary_sort_ties'),
        'pagination_warning': info.get('pagination_warning')})
print('Verified quarters in research scope:',len(verified_quarters),'/',len(ALL_QUARTERS))
print('Verified in CURRENT RUN:',len(set(TARGET_QUARTERS)&set(verified_quarters)),'/',len(TARGET_QUARTERS))
print(pd.DataFrame(source_data).to_string(index=False))

Validating cached Parquet: 1/170 | Quarter: 19840331
Validating cached Parquet: 10/170 | Quarter: 19860630
Validating cached Parquet: 20/170 | Quarter: 19881231
Validating cached Parquet: 30/170 | Quarter: 19910630
Validating cached Parquet: 40/170 | Quarter: 19931231
Validating cached Parquet: 50/170 | Quarter: 19960630
Validating cached Parquet: 60/170 | Quarter: 19981231
Validating cached Parquet: 70/170 | Quarter: 20010630
Validating cached Parquet: 80/170 | Quarter: 20031231
Validating cached Parquet: 90/170 | Quarter: 20060630
Validating cached Parquet: 100/170 | Quarter: 20081231
Validating cached Parquet: 110/170 | Quarter: 20110630
Validating cached Parquet: 120/170 | Quarter: 20131231
Validating cached Parquet: 130/170 | Quarter: 20160630
Validating cached Parquet: 140/170 | Quarter: 20181231
Validating cached Parquet: 150/170 | Quarter: 20210630
Validating cached Parquet: 160/170 | Quarter: 20231231
Validating cached Parquet: 170/170 | Quarter: 20260630
Verified quarters in 

## 10. Schema audit + Parquet storage layout

Raw landing: `data_raw/fdic/financials/REPDTE=YYYYMMDD.parquet`. Curated year partitions: `data_storage/financials/year_partitioned_pilot|full/YEAR=YYYY/data.parquet`. Raw quarterly files giữ nguyên cho resume. Curated build từ verified raw file, không impute null và không trộn failure label. Dữ liệu được lưu trên **Drive**.

In [ ]:
# Only verified raw quarters of the CURRENT mode are included. No full-history concat.
mode_verified = [q for q in TARGET_QUARTERS if q in verified_quarters]
schema_audit = []
curated_inventory = []
verified_rows_current_mode = 0

if mode_verified:
    # A year's data (maximum 4 quarterly files) is small enough to materialize.
    # We never materialize all 170 quarters merely to build schema/curated storage.
    CURATED.mkdir(parents=True, exist_ok=True)
    for child in CURATED.glob('YEAR=*/data.parquet'):
        child.unlink()
    field_null_counts = {field.name: 0 for field in SCHEMAS['financials']}
    years = sorted(set(q[:4] for q in mode_verified))
    for yr in years:
        qs = [q for q in mode_verified if q[:4] == yr]
        pieces = [pq.ParquetFile(financial_path(q)).read() for q in qs]
        part = pa.concat_tables(pieces, promote_options='none')
        for name in field_null_counts:
            field_null_counts[name] += part.column(name).null_count
        target = CURATED / f'YEAR={yr}' / 'data.parquet'
        write_verified_parquet(part, target)
        curated_inventory.append({'year': int(yr), 'quarters': len(qs),
                                  'rows': part.num_rows, 'bytes': target.stat().st_size,
                                  'file': str(target.relative_to(PROJECT_ROOT))})
        verified_rows_current_mode += part.num_rows
        del pieces, part
    gc.collect()
    for field in SCHEMAS['financials']:
        missing = field_null_counts[field.name]
        schema_audit.append({'field': field.name, 'arrow_type': str(field.type),
                             'rows': verified_rows_current_mode, 'null_count': missing,
                             'null_pct': round(100 * missing / verified_rows_current_mode, 4)})
    expected_rows = sum(int(manifest['entries']['financials:' + q]['rows']) for q in mode_verified)
    assert expected_rows == verified_rows_current_mode, 'Curated row count mismatch'
    print(f'Curated: {len(curated_inventory)} years / {verified_rows_current_mode:,} rows; '
          'streamed one year at a time')
else:
    print('Curated skipped: no verified quarters in current mode.')

Curated: 43 years / 1,682,616 rows; streamed one year at a time


## 11. Fair CSV vs Parquet benchmark — same PyArrow engine, local Colab disk

**Đo cả write/read/projection/filter** theo 5 lần lặp lấy median, 1 warm-up. Cả hai file được sinh từ **cùng Arrow table**, `CSV` được đọc với Arrow `ConvertOptions` khớp schema, `Parquet` dùng PyArrow dataset filter pushdown. Kiểm tra nội dung kết quả lọc (không chỉ đếm dòng).

Đo trên `/content` để tránh Drive I/O; report ghi cấu hình **Colab runtime**, không gán sang laptop cá nhân. Với FULL, giới hạn sample (`BENCHMARK_MAX_ROWS`) nếu Colab thiếu RAM và phải công bố sample trong report. Nếu để `None`, notebook **tự ước lượng** bộ nhớ từ Parquet metadata, dùng ngân sách RAM cấu hình và lấy mẫu theo tỷ trọng số dòng của từng quý với random seed cố định. Báo cáo ghi `scope=ALL_VERIFIED` hoặc `SAMPLE`.

In [ ]:
def bench_ms(fn, repetitions=BENCHMARK_REPEATS):
    fn()  # warm-up
    vals = []
    for _ in range(repetitions):
        start = time.perf_counter()
        fn()
        vals.append((time.perf_counter() - start) * 1000)
    return round(statistics.median(vals), 3)


def build_memory_safe_benchmark_table(quarters):
    if not quarters:
        return None, {'scope': 'NO_DATA', 'rows_available': 0, 'rows_benchmarked': 0}

    file_stats = []
    for q in quarters:
        pf = pq.ParquetFile(financial_path(q))
        est_uncompressed = sum(pf.metadata.row_group(j).total_byte_size
                               for j in range(pf.metadata.num_row_groups))
        file_stats.append((q, pf.metadata.num_rows, max(est_uncompressed, 1)))
    total_rows = sum(n for _, n, _ in file_stats)
    estimated_bytes = sum(size for _, _, size in file_stats)

    try:
        import psutil
        # Leave headroom for the Colab runtime and other objects.
        available_bytes = int(psutil.virtual_memory().available * 0.35)
    except ImportError:
        available_bytes = int(BENCHMARK_MEMORY_BUDGET_GIB * 1024**3)
    configured_bytes = int(BENCHMARK_MEMORY_BUDGET_GIB * 1024**3)
    budget_bytes = min(configured_bytes, available_bytes)
    conservative_estimate = estimated_bytes * BENCHMARK_RAM_SAFETY_FACTOR
    auto_cap = (total_rows if conservative_estimate <= budget_bytes else
                max(1, int(total_rows * budget_bytes / conservative_estimate)))
    cap = min(total_rows, auto_cap,
              BENCHMARK_MAX_ROWS if BENCHMARK_MAX_ROWS is not None else total_rows)
    is_sample = cap < total_rows

    # Allocate sampling quota across all quarters, proportional to actual row counts,
    # rather than taking the first N rows of the oldest year only.
    weighted = [n * cap / total_rows for _, n, _ in file_stats]
    alloc = [min(n, int(weight)) for (_, n, _), weight in zip(file_stats, weighted)]
    remaining = cap - sum(alloc)
    order = sorted(range(len(file_stats)),
                   key=lambda i: (weighted[i] - alloc[i], -i), reverse=True)
    for i in order:
        if remaining <= 0:
            break
        if alloc[i] < file_stats[i][1]:
            alloc[i] += 1
            remaining -= 1
    assert sum(alloc) == cap

    pieces = []
    for (q, expected_rows, _), keep in zip(file_stats, alloc):
        if keep == 0:
            continue
        part = pq.ParquetFile(financial_path(q)).read()
        if part.num_rows != expected_rows:
            raise RuntimeError(f'Quarter changed during benchmark: {q}')
        if keep < expected_rows:
            rng = random.Random(BENCHMARK_RANDOM_SEED + int(q))
            selected = sorted(rng.sample(range(expected_rows), keep))
            part = part.take(pa.array(selected, type=pa.int64()))
        pieces.append(part)
    tab = pa.concat_tables(pieces, promote_options='none')
    del pieces
    gc.collect()

    meta = {'scope': 'SAMPLE' if is_sample else 'ALL_VERIFIED',
            'rows_available': total_rows, 'rows_benchmarked': tab.num_rows,
            'quarters_available': len(quarters),
            'approx_uncompressed_mib': round(estimated_bytes / 1024**2, 2),
            'conservative_ram_estimate_mib': round(conservative_estimate / 1024**2, 2),
            'memory_budget_mib': round(budget_bytes / 1024**2, 2),
            'manual_max_rows': BENCHMARK_MAX_ROWS,
            'random_seed': BENCHMARK_RANDOM_SEED,
            'sampling_method': 'proportional-per-quarter uniform random, fixed seed' if is_sample else 'none'}
    print('Benchmark scope:', meta)
    return tab, meta


storage_bench = []
codec_bench = []
partition_bench = []
benchmark_meta = {'scope': 'NOT_RUN', 'rows_available': verified_rows_current_mode,
                  'rows_benchmarked': 0}
tab = None
if RUN_BENCHMARK and mode_verified:
    LOCAL_BENCH.mkdir(parents=True, exist_ok=True)
    tab, benchmark_meta = build_memory_safe_benchmark_table(mode_verified)
    csv_dest = LOCAL_BENCH / 'financials_benchmark.csv'
    pq_dest = LOCAL_BENCH / 'financials_snappy.parquet'
    pacsv.write_csv(tab, csv_dest)
    pq.write_table(tab, pq_dest, compression='snappy', version='2.6')
    convert = pacsv.ConvertOptions(column_types=tab.schema)
    csv_load = lambda: pacsv.read_csv(csv_dest, convert_options=convert)
    pq_load = lambda: pq.read_table(pq_dest)
    parquet_ds = ds.dataset(str(pq_dest), format='parquet')
    cols = [c for c in ('CERT', 'REPDTE', 'ASSET') if c in tab.column_names]
    filter_q = tab.column('REPDTE')[0].as_py()
    predicate = ds.field('REPDTE') == filter_q

    def csv_filtered():
        data = csv_load()
        return data.filter(pc.equal(data.column('REPDTE'), filter_q)).select(cols)

    parquet_filtered = lambda: parquet_ds.to_table(columns=cols, filter=predicate)
    assert csv_filtered().sort_by([('CERT', 'ascending')]).equals(
        parquet_filtered().sort_by([('CERT', 'ascending')]),
        check_metadata=False), 'CSV/Parquet filtered content mismatch'

    experiments = [
        ('CSV', csv_dest, lambda: pacsv.write_csv(tab, csv_dest), csv_load,
         lambda: csv_load().select(cols), csv_filtered),
        ('PARQUET_SNAPPY', pq_dest,
         lambda: pq.write_table(tab, pq_dest, compression='snappy', version='2.6'),
         pq_load, lambda: parquet_ds.to_table(columns=cols), parquet_filtered)
    ]
    for label, path, writer, reader, projecter, filtered in experiments:
        storage_bench.append({
            'format': label, 'engine': 'PyArrow', 'run_mode': RUN_MODE,
            'scope': benchmark_meta['scope'], 'rows': tab.num_rows,
            'total_verified_rows': benchmark_meta['rows_available'],
            'columns': tab.num_columns, 'size_mib': round(path.stat().st_size / 1024**2, 4),
            'write_ms_median': bench_ms(writer), 'full_read_ms_median': bench_ms(reader),
            'project_3_cols_ms_median': bench_ms(projecter),
            'filter_ms_median': bench_ms(filtered), 'filter_quarter': filter_q,
            'filter_rows': csv_filtered().num_rows,
            'repeats': BENCHMARK_REPEATS, 'storage': 'Colab local /content'
        })
    print(pd.DataFrame(storage_bench).to_string(index=False))
else:
    print('Storage benchmark skipped: no verified data or RUN_BENCHMARK=False')

Benchmark scope: {'scope': 'ALL_VERIFIED', 'rows_available': 1682616, 'rows_benchmarked': 1682616, 'quarters_available': 170, 'approx_uncompressed_mib': 428.71, 'conservative_ram_estimate_mib': 1286.13, 'memory_budget_mib': 1536.0, 'manual_max_rows': None, 'random_seed': 42, 'sampling_method': 'none'}
        format  engine run_mode        scope    rows  total_verified_rows  columns  size_mib  write_ms_median  full_read_ms_median  project_3_cols_ms_median  filter_ms_median filter_quarter  filter_rows  repeats              storage
           CSV PyArrow     FULL ALL_VERIFIED 1682616              1682616       47  439.6493        13047.654             4080.122                  4560.610          4348.269       19840331        17930        5 Colab local /content
PARQUET_SNAPPY PyArrow     FULL ALL_VERIFIED 1682616              1682616       47  221.3441         5365.200              860.896                   101.242            67.413       19840331        17930        5 Colab local /conten

## 12. Compression & partitioning benchmarks

`NONE` / `SNAPPY` / `GZIP` / `ZSTD` trên cùng Arrow table. Partition benchmark so sánh monolithic vs Hive partitioned trên cùng query và kiểm tra **giá trị kết quả**. Pilot chia theo quý (`REPDTE`) để có 4 partitions; FULL chia theo năm (`YEAR`) phù hợp layout đã chọn.

In [ ]:
if tab is not None:
    for name, codec in [('NONE',None),('SNAPPY','snappy'),('GZIP','gzip'),('ZSTD','zstd')]:
        path=LOCAL_BENCH/f'codec_{name.lower()}.parquet'
        writer=lambda p=path,c=codec: pq.write_table(tab,p,compression=c,version='2.6')
        w=bench_ms(writer)
        r=bench_ms(lambda p=path: pq.read_table(p))
        codec_bench.append({'codec':name,'size_mib':round(path.stat().st_size/1024**2,4),
                            'write_ms_median':w,'read_ms_median':r,
                            'rows':tab.num_rows,'repeats':BENCHMARK_REPEATS,
                            'run_mode':RUN_MODE,'benchmark_scope':benchmark_meta['scope'],
                            'total_verified_rows':benchmark_meta['rows_available'],
                            'storage':'Colab local /content'})

    partition_key='REPDTE' if RUN_MODE=='PILOT' else 'YEAR'
    data_year=pa.array([int(str(v)[:4]) for v in tab.column('REPDTE').to_pylist()],pa.int32())
    with_year=tab.append_column('YEAR',data_year)
    mono=LOCAL_BENCH/'partition_monolithic.parquet'
    partition_root=LOCAL_BENCH/f'partition_by_{partition_key.lower()}'
    if partition_root.exists(): shutil.rmtree(partition_root)
    partition_root.mkdir(parents=True)
    pq.write_table(with_year,mono,compression='snappy',version='2.6')
    for key in sorted(set(with_year.column(partition_key).to_pylist())):
        fragment=with_year.filter(pc.equal(with_year.column(partition_key),key)).drop([partition_key])
        dst=partition_root/f'{partition_key}={key}'/'data.parquet'
        dst.parent.mkdir(parents=True,exist_ok=True)
        pq.write_table(fragment,dst,compression='snappy',version='2.6')
    scheme=ds.partitioning(pa.schema([(partition_key,pa.string() if partition_key=='REPDTE' else pa.int32())]),flavor='hive')
    mono_ds=ds.dataset(str(mono),format='parquet')
    part_ds=ds.dataset(str(partition_root),format='parquet',partitioning=scheme)
    key_value=tab.column('REPDTE')[0].as_py() if partition_key=='REPDTE' else data_year[0].as_py()
    pred=ds.field(partition_key)==key_value
    cols=['CERT','REPDTE','ASSET']
    mono_fn=lambda:mono_ds.to_table(columns=cols,filter=pred)
    part_fn=lambda:part_ds.to_table(columns=cols,filter=pred)
    left=mono_fn().sort_by([('CERT','ascending'),('REPDTE','ascending')]).combine_chunks()
    right=part_fn().sort_by([('CERT','ascending'),('REPDTE','ascending')]).combine_chunks()
    assert left.equals(right,check_metadata=False), 'Hive partition query result mismatch'
    for label,path,fn in [('MONOLITHIC',mono,mono_fn),('HIVE_PARTITIONED',partition_root,part_fn)]:
        bytes_size=path.stat().st_size if path.is_file() else sum(p.stat().st_size for p in path.rglob('*.parquet'))
        partition_bench.append({'layout':label,'partition_column':partition_key,
            'predicate':f'{partition_key} == {key_value}',
            'matched_rows':left.num_rows,'value_parity':True,
            'size_mib':round(bytes_size/1024**2,4),
            'filter_ms_median':bench_ms(fn), 'repeats':BENCHMARK_REPEATS,
            'run_mode':RUN_MODE,'benchmark_scope':benchmark_meta['scope'],
                            'total_verified_rows':benchmark_meta['rows_available'],
                            'storage':'Colab local /content'})
    print('Compression / Partition benchmark complete:',len(codec_bench),len(partition_bench))

Compression / Partition benchmark complete: 5 2


### One-time repair: Failures + Institutions API IDs (no Financials acquisition)

Run this **only after** the revised `acquire_aux()` definition cell has been run in the **same Colab runtime**.
It re-downloads only invalid auxiliary pages, then checks every API_RECORD_ID against the
new snapshot Parquet files and updates the in-memory source/live-audit reports.
**Do not run the original "Execute acquisition" cell** to apply this fix.

After this one-time cell completes, run **only** the existing
**"Export all Part 1 reports"** cell and then **"Acceptance criteria & interpretation"**
cell. Existing Financials Parquet and curated partitions are not rewritten.

If the Colab runtime has restarted, first run the setup, schema, class/function-definition
and manifest-loading cells needed to restore the variables; do **not** rerun acquisition.
The existing 170 Financials audit rows are intentionally reused, rather than requesting
170 live totals again. Re-run the original live audit separately for an entirely new check.


In [ ]:
# ONE-TIME AUXILIARY REPAIR — DO NOT RUN THE FULL ACQUISITION CELL.
# Prerequisite: rerun the revised acquire_aux() DEFINITION cell first.
# This cell writes only data_raw/fdic/aux_pages/{failures,institutions}
# and updates their manifest records. It does NOT call acquire_financial().

assert RUN_MODE == 'FULL', 'Keep RUN_MODE=FULL for the existing 170-quarter project.'
assert callable(acquire_aux), 'Run the revised acquire_aux definition cell first.'
assert len(TARGET_QUARTERS) == 170, 'Unexpected quarterly research scope.'
assert len([x for x in source_data if x['source'] == 'financials']) == 1
assert len([x for x in post_download_api_audit if x['source'] == 'financials']) == 170, (
    'Financials post-download audit is missing from memory; load/run the prior audit cell first.'
)

AUX_REPAIR_SOURCES = ('failures', 'institutions')
financial_manifest_before = {
    k: (v.get('status'), v.get('rows'), v.get('sha256'))
    for k, v in manifest['entries'].items()
    if k.startswith('financials:')
}
assert len(financial_manifest_before) == 170, 'Financials manifest must already have 170 quarters.'

for src in AUX_REPAIR_SOURCES:
    print(f'\n=== Repairing {src.upper()} only ===', flush=True)
    result = acquire_aux(src)
    if (result.get('status') not in {'VERIFIED', 'VERIFIED_SCHEMA_GAP'}
            or result.get('api_id_check') != 'PASS'
            or result.get('missing_api_ids') != 0
            or result.get('duplicate_api_ids') != 0
            or result.get('rows') != result.get('api_total')):
        raise RuntimeError(f'{src} did not pass full acquisition/ID verification: {result}')

    # Independent readback across CURRENT snapshot pages, not stale snapshots.
    ids = []
    for page_idx in range(result['pages']):
        key = f'aux:{src}:{result["snapshot"]}:{page_idx:05d}'
        page_entry = manifest['entries'].get(key)
        page_path = aux_page_path(src, result['snapshot'], page_idx)
        valid, reason = verified_cache(page_entry, page_path, src)
        if not valid:
            raise RuntimeError(f'{src} current snapshot page {page_idx} invalid: {reason}')
        ids.extend(pq.read_table(page_path, columns=['API_RECORD_ID'])
                   .column('API_RECORD_ID').to_pylist())

    missing = sum(v is None or not str(v).strip() for v in ids)
    unique_count = len(set(v for v in ids if v is not None and str(v).strip()))
    duplicate_count = len(ids) - missing - unique_count
    assert len(ids) == result['api_total'], f'{src}: reconstructed row count mismatch'
    assert missing == 0 and duplicate_count == 0, (
        f'{src}: readback ID audit failed: missing={missing}, duplicates={duplicate_count}'
    )
    print(f'READBACK PASS {src}: rows={len(ids):,}, unique IDs={unique_count:,}, '
          'missing=0, duplicates=0', flush=True)

financial_manifest_after = {
    k: (v.get('status'), v.get('rows'), v.get('sha256'))
    for k, v in manifest['entries'].items()
    if k.startswith('financials:')
}
assert financial_manifest_after == financial_manifest_before, (
    'Unexpected change to Financials manifest; investigate before proceeding.'
)
print('Financials manifest: 170 quarterly entries unchanged.')

# Update existing in-memory source completeness ONLY for auxiliary sources.
source_data[:] = [r for r in source_data if r.get('source') not in AUX_REPAIR_SOURCES]
for src in AUX_REPAIR_SOURCES:
    info = manifest['entries'][f'aux:{src}:summary']
    source_data.append({
        'source': src, 'scope': 'entire endpoint',
        'api_total_at_download': info.get('api_total'),
        'downloaded_rows': info.get('rows'),
        'target_quarters': None, 'verified_quarters': None,
        'status': info.get('status'), 'api_id_check': info.get('api_id_check'),
        'unique_api_ids': info.get('unique_api_ids'),
        'duplicate_api_ids': info.get('duplicate_api_ids'),
        'missing_api_ids': info.get('missing_api_ids'),
        'page_boundary_sort_ties': info.get('page_boundary_sort_ties'),
        'pagination_warning': info.get('pagination_warning')
    })

# Refresh live totals for ONLY Failures and Institutions. Preserve the
# existing 170 Financials live-audit rows from the successful earlier run.
post_download_api_audit[:] = [
    r for r in post_download_api_audit if r.get('source') not in AUX_REPAIR_SOURCES
]
for src in AUX_REPAIR_SOURCES:
    info = manifest['entries'][f'aux:{src}:summary']
    row = {
        'source': src, 'quarter': '', 'manifest_status': info['status'],
        'manifest_api_total': info['api_total'], 'local_rows': info['rows'],
        'live_api_total': None, 'audit_result': 'NOT_RUN',
        'error': '', 'checked_at': None
    }
    if RUN_POST_DOWNLOAD_API_AUDIT:
        try:
            _, live_total = api.page(src, [FIELDS[src][0]], sort_by=SORT_KEYS[src], limit=1)
            row['live_api_total'] = live_total
            row['audit_result'] = ('MATCH' if live_total == info['api_total'] == info['rows']
                                   else 'DRIFT')
        except Exception as exc:
            row['audit_result'] = 'ERROR'
            row['error'] = f'{type(exc).__name__}: {str(exc)[:200]}'
        row['checked_at'] = utc_now()
    post_download_api_audit.append(row)

# Existing Financials audit is reused, not rerun. This also updates G7 inputs.
audit_df = pd.DataFrame(post_download_api_audit)
post_audit_all_match = (
    bool(RUN_POST_DOWNLOAD_API_AUDIT) and not audit_df.empty
    and bool(audit_df['audit_result'].eq('MATCH').all())
)
print('Updated audit counts:', audit_df['audit_result'].value_counts().to_dict())
print('Post-download audit all match:', post_audit_all_match)
print(pd.DataFrame(source_data).to_string(index=False))
print('\nNEXT: run only the existing "Export all Part 1 reports" cell, '
      'then "Acceptance criteria & interpretation" cell.')



=== Repairing FAILURES only ===
failures: API total=4,117; pages=1; snapshot=6c84e6f95b003ba9da
  failures: 1/1 pages; 4,117 rows
failures: VERIFIED 4,117/4,117 rows; IDs unique=4,117, duplicates=0, missing=0, sort-boundary ties=0
READBACK PASS failures: rows=4,117, unique IDs=4,117, missing=0, duplicates=0

=== Repairing INSTITUTIONS only ===
institutions: API total=27,835; pages=6; snapshot=9ddae3c7e5fbbe963c
  institutions: 6/6 pages; 27,835 rows
institutions: VERIFIED 27,835/27,835 rows; IDs unique=27,835, duplicates=0, missing=0, sort-boundary ties=0
READBACK PASS institutions: rows=27,835, unique IDs=27,835, missing=0, duplicates=0
Financials manifest: 170 quarterly entries unchanged.
Updated audit counts: {'MATCH': 172}
Post-download audit all match: True
      source           scope  api_total_at_download  downloaded_rows  target_quarters  verified_quarters   status api_id_check  unique_api_ids  duplicate_api_ids  missing_api_ids  page_boundary_sort_ties pagination_warning
  f

## 13. Export all Part 1 reports & acceptance gates to Drive

Bảng xuất: `download_manifest.csv`, `quarter_coverage.csv`, `record_reconciliation.csv`, `source_completeness.csv`, `schema_audit.csv`, `storage_benchmark.csv`, `compression_benchmark.csv`, `partition_benchmark.csv`, `curated_year_manifest.csv`, `schema_report.json`, `acceptance_gates.csv`, `post_download_api_audit.csv`, `acquisition_summary.json`. **Reports được ghi từ trạng thái hiện tại**, không trộn với outputs đã được lưu trong Notebook cũ.

In [ ]:
def export_csv(data,name,cols=None):
    frame=pd.DataFrame(data)
    if frame.empty and cols is not None:
        frame=pd.DataFrame(columns=cols)
    target=REPORTS/name
    frame.to_csv(target,index=False,encoding='utf-8-sig')
    print('Saved:',target.relative_to(PROJECT_ROOT),'rows=',len(frame))

manifest_rows=[]
for key,val in sorted(manifest['entries'].items()):
    manifest_rows.append({'task':key,**{k:(';'.join(map(str,v)) if isinstance(v,list) else v)
                                         for k,v in val.items()}})
export_csv(manifest_rows,'download_manifest.csv')
export_csv(coverage,'quarter_coverage.csv')
export_csv(reconciliation,'record_reconciliation.csv')
export_csv(source_data,'source_completeness.csv')
export_csv(post_download_api_audit, 'post_download_api_audit.csv')
export_csv(schema_audit,'schema_audit.csv',cols=['field','arrow_type','rows','null_count','null_pct'])
export_csv(storage_bench,'storage_benchmark.csv',cols=['format','engine','rows','size_mib'])
export_csv(codec_bench,'compression_benchmark.csv',cols=['codec','size_mib'])
export_csv(partition_bench,'partition_benchmark.csv',cols=['layout','partition_column','filter_ms_median'])
export_csv(curated_inventory,'curated_year_manifest.csv',cols=['year','quarters','rows','bytes','file'])

try:
    import psutil
    ram_gib=round(psutil.virtual_memory().total/1024**3,2)
except ImportError:
    ram_gib=None
hardware={'platform':platform.platform(),'python':sys.version.split()[0],
          'pandas':pd.__version__,'pyarrow':pa.__version__,
          'logical_cpu':os.cpu_count(),'ram_gib':ram_gib,'benchmark_disk':'/content temporary local disk'}
schema_report={'run_mode':RUN_MODE,'generated_at':utc_now(),
    'dictionary':str(DICT_PATH),'dictionary_sha256':DICTIONARY_HASH,
    'fields':FIELDS,
    'arrow_types':{s:{f.name:str(f.type) for f in SCHEMAS[s]} for s in DATASETS},
    'signatures':SIGNATURES,'hardware':hardware,
    'raw_layout':'data_raw/fdic/financials/REPDTE=YYYYMMDD.parquet (Drive, Snappy)',
    'curated_layout':f'data_storage/financials/year_partitioned_{RUN_MODE.lower()}/YEAR=YYYY/data.parquet (Drive, Snappy)',
    'benchmark_disk':'/content/fdic_part1_bench (temporary, local, not Drive)',
    'benchmark_selection': benchmark_meta,
    'post_download_audit_enabled': RUN_POST_DOWNLOAD_API_AUDIT,
    'integer_validation': 'integer YAML → Arrow int64; Decimal exact-integrality check; no float truncation',
    'aux_record_id_validation': 'Nested FDIC data.ID stored as API_RECORD_ID; full cross-page duplicate/missing audit',
    'field_gap_rule':'Missing fields observed in API responses are reported, not treated as fully available',
    'completeness_rule':'Rows equal captured API meta.total and files pass local hash, schema, key/date audit'}
save_json_atomic(schema_report, REPORTS/'schema_report.json')

Saved: outputs/part1/download_manifest.csv rows= 186
Saved: outputs/part1/quarter_coverage.csv rows= 170
Saved: outputs/part1/record_reconciliation.csv rows= 170
Saved: outputs/part1/source_completeness.csv rows= 3
Saved: outputs/part1/post_download_api_audit.csv rows= 172
Saved: outputs/part1/schema_audit.csv rows= 47
Saved: outputs/part1/storage_benchmark.csv rows= 2
Saved: outputs/part1/compression_benchmark.csv rows= 5
Saved: outputs/part1/partition_benchmark.csv rows= 2
Saved: outputs/part1/curated_year_manifest.csv rows= 43


## 14. Acceptance criteria & interpretation

**PILOT PASS** nếu cả 4 quý hoàn thành record/key checks, full Failures + Institutions đạt record completeness, và benchmark chạy. **FULL record coverage PASS** nếu đủ 170 quý *và* full Failures + Institutions, cùng lượt API total audit (bật ở FULL) không có drift. `SCHEMA_GAP` vẫn cần giải thích khi viết report; không được nói “47/47 fields complete” nếu API bỏ trường ở những năm trước.

Nếu Colab ngắt trong download, reports có thể chưa được ghi; manifest checkpoint vẫn tồn tại và chạy lại sẽ skip dữ liệu tốt.

In [ ]:
mode_coverage = coverage_df[coverage_df.requested_in_this_run]
okay_quarters = int(mode_coverage.status.isin(['VERIFIED', 'VERIFIED_SCHEMA_GAP']).sum())
all_quarters_verified = (okay_quarters == len(TARGET_QUARTERS))
aux_info = {s: manifest['entries'].get(f'aux:{s}:summary', {})
            for s in ('failures', 'institutions')}
aux_status = {s: info.get('status', 'NOT_ATTEMPTED') for s, info in aux_info.items()}
aux_record_complete = all(v in {'VERIFIED', 'VERIFIED_SCHEMA_GAP'} for v in aux_status.values())
aux_ids_verified = all(info.get('api_id_check') == 'PASS'
                       and info.get('duplicate_api_ids') == 0
                       and info.get('missing_api_ids') == 0
                       for info in aux_info.values())
strict_schema_complete = (bool(len(mode_coverage))
                          and bool((mode_coverage.status == 'VERIFIED').all())
                          and all(v == 'VERIFIED' for v in aux_status.values()))
measured = bool(storage_bench and codec_bench and partition_bench)
raw_and_curated = (verified_rows_current_mode > 0
                   and sum(x['rows'] for x in curated_inventory) == verified_rows_current_mode)
full_complete = (RUN_MODE == 'FULL' and len(ALL_QUARTERS) == 170
                 and all_quarters_verified and aux_record_complete and aux_ids_verified
                 and post_audit_all_match)

gates = [
 {'gate': 'G1_financial_quarters_record_verified', 'passed': all_quarters_verified,
  'detail': f'{okay_quarters}/{len(TARGET_QUARTERS)} target quarters'},
 {'gate': 'G2_failures_and_institutions_complete', 'passed': aux_record_complete,
  'detail': str(aux_status)},
 {'gate': 'G3_aux_API_record_IDs_unique_and_present', 'passed': aux_ids_verified,
  'detail': str({s: {k: info.get(k) for k in ('api_id_check', 'duplicate_api_ids', 'missing_api_ids')}
                 for s, info in aux_info.items()})},
 {'gate': 'G4_all_selected_fields_observed', 'passed': strict_schema_complete,
  'detail': 'Historical field gaps are reported separately from record completeness'},
 {'gate': 'G5_curated_rows_match_raw', 'passed': bool(raw_and_curated),
  'detail': f'{len(curated_inventory)} year partitions'},
 {'gate': 'G6_all_three_benchmarks_measured', 'passed': measured,
  'detail': f'{len(storage_bench)} storage / {len(codec_bench)} codecs / {len(partition_bench)} layouts; '
            f"scope={benchmark_meta['scope']}"},
 {'gate': 'G7_live_API_total_post_audit', 'passed': (bool(post_audit_all_match) if RUN_POST_DOWNLOAD_API_AUDIT else None),
  'detail': ('Live audit counts: ' + str(audit_df['audit_result'].value_counts().to_dict())
             if RUN_POST_DOWNLOAD_API_AUDIT else 'Not run: audit disabled')},
 {'gate': 'FULL_RECORD_COVERAGE_VERIFIED', 'passed': bool(full_complete),
  'detail': 'FULL only; requires verified ID, acquisition records, and live audit when enabled'}
]
export_csv(gates, 'acceptance_gates.csv')

summary = {'run_mode': RUN_MODE, 'generated_at': utc_now(),
 'research_quarters': len(ALL_QUARTERS), 'target_quarters': len(TARGET_QUARTERS),
 'verified_quarters': okay_quarters,
 'strict_all_fields_quarters': int((mode_coverage.status == 'VERIFIED').sum()),
 'source_gap_quarters': mode_coverage.loc[mode_coverage.status == 'SOURCE_GAP', 'quarter'].tolist(),
 'schema_gap_quarters': mode_coverage.loc[mode_coverage.status == 'VERIFIED_SCHEMA_GAP', 'quarter'].tolist(),
 'not_verified_quarters': mode_coverage.loc[
     ~mode_coverage.status.isin(['VERIFIED', 'VERIFIED_SCHEMA_GAP']), 'quarter'].tolist(),
 'verified_rows_current_mode': verified_rows_current_mode,
 'failures_and_institutions': aux_status,
 'aux_id_audit': {s: {k: info.get(k) for k in
    ('api_id_check', 'unique_api_ids', 'duplicate_api_ids',
     'missing_api_ids', 'page_boundary_sort_ties')} for s, info in aux_info.items()},
 'post_download_api_audit_enabled': RUN_POST_DOWNLOAD_API_AUDIT,
 'post_download_api_audit_all_match': bool(post_audit_all_match),
 'post_download_api_audit_result_counts': audit_df['audit_result'].value_counts().to_dict(),
 'full_record_coverage_verified': bool(full_complete),
 'strict_all_selected_fields_verified': bool(strict_schema_complete),
 'benchmark': benchmark_meta,
 'limitations': ['FDIC may revise records without changing API total',
                 'Offset pagination with a non-unique sort key is not a guaranteed snapshot',
                 'Not all modern field definitions exist in early historical quarters',
                 'Colab local benchmark disk is not the laptop or Google Drive']}
save_json_atomic(summary, REPORTS / 'acquisition_summary.json')
print(pd.DataFrame(gates).to_string(index=False))
print('\nFULL_RECORD_COVERAGE_VERIFIED:', bool(full_complete))
print('Reports directory:', REPORTS)
if RUN_MODE == 'PILOT':
    print('PILOT finished. Review acceptance gates before changing RUN_MODE to FULL.')
else:
    print('FULL finished. Review any schema gaps, ID issues, audit drift and benchmark scope.')

Saved: outputs/part1/acceptance_gates.csv rows= 8
                                    gate  passed                                                                                                                                                                       detail
   G1_financial_quarters_record_verified    True                                                                                                                                                      170/170 target quarters
   G2_failures_and_institutions_complete    True                                                                                                                         {'failures': 'VERIFIED', 'institutions': 'VERIFIED'}
G3_aux_API_record_IDs_unique_and_present    True {'failures': {'api_id_check': 'PASS', 'duplicate_api_ids': 0, 'missing_api_ids': 0}, 'institutions': {'api_id_check': 'PASS', 'duplicate_api_ids': 0, 'missing_api_ids': 0}}
         G4_all_selected_fields_observed    True              

### Auxiliary Data Integrity Validation

This validation assesses the completeness and uniqueness of API record identifiers in the cached FDIC Failures and Institutions datasets.

For each source, the notebook verifies:

- Total downloaded record count.
- Number of valid API record identifiers.
- Number of missing API record identifiers.
- Number of duplicate API record identifiers.

**Validation results:**

| Dataset | Total Records | Missing IDs | Duplicate IDs |
|---|---:|---:|---:|
| Failures | 4,117 | 0 | 0 |
| Institutions | 27,835 | 0 | 0 |

All 31,952 records contain non-missing API identifiers, with no duplicate identifiers detected within either dataset.

These checks strengthen the acquisition integrity assessment and support subsequent reconciliation against FDIC API totals.

In [ ]:
import pyarrow.parquet as pq

for source in ["failures", "institutions"]:
    snapshot = manifest["entries"].get(f"aux:{source}:summary", {}).get("snapshot")
    keys = [
        k for k in manifest["entries"]
        if snapshot and k.startswith(f"aux:{source}:{snapshot}:")
    ]

    all_ids = []
    total_rows = 0

    for key in sorted(keys):
        entry = manifest["entries"][key]
        path = PROJECT_ROOT / entry["file"]

        table = pq.read_table(path)

        total_rows += table.num_rows
        all_ids.extend(
            table.column("API_RECORD_ID").to_pylist()
        )

    valid_ids = [
        x for x in all_ids
        if x is not None and x != ""
    ]

    print(f"\n{source.upper()}")
    print("Total rows:", total_rows)
    print("Valid API IDs:", len(valid_ids))
    print("Missing API IDs:", total_rows - len(valid_ids))
    print("Duplicate API IDs:", len(valid_ids) - len(set(valid_ids)))


FAILURES
Total rows: 4117
Valid API IDs: 4117
Missing API IDs: 0
Duplicate API IDs: 0

INSTITUTIONS
Total rows: 27835
Valid API IDs: 27835
Missing API IDs: 0
Duplicate API IDs: 0


### Part 1 report & reproducibility checklist

1. Mô tả phương pháp FDIC BankFind API, schema từ Notebook 01 và thời điểm lấy dữ liệu (`manifest_colab_v1.json`).
2. Báo cáo **số dòng thực tế**, số quý đã kiểm tra, số quý `SOURCE_GAP`, `VERIFIED_SCHEMA_GAP`; nếu FULL chưa đạt, công bố hạn chế thay vì tuyên bố đầy đủ.
3. Trích `post_download_api_audit.csv`, `source_completeness.csv` (bao gồm kiểm tra API_RECORD_ID) khi khẳng định độ đầy đủ.
4. Bảng so sánh kích thước và median latency của CSV vs Parquet, codecs, layout; chỉ dùng các file CSV xuất trong lần chạy của bạn. PILOT số liệu nhỏ không được suy rộng cho FULL.
5. Benchmark trên Colab VM (Python, CPU, RAM, PyArrow, local disk); nêu cấu hình trong `schema_report.json`.
6. GitHub chỉ đẩy notebooks, `README.md`, pinned `requirements.txt`, figures không chứa dữ liệu thô. Không đẩy `/data_raw`, `/data_storage`, hoặc credentials/manifest chứa chi tiết truy cập.
7. Part 2 mới tạo clean bank-quarter panel, phân biệt failures vs other exits. Part 4–5 mới tạo nhãn future 4-quarter và split thời gian; **không sử dụng thông tin tương lai trong Part 1 data acquisition**.